In [1]:
!apt-get update -qq

!apt-get install openjdk-8-jdk-headless -qq > /dev/null

!wget https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz

!tar -xzf hadoop-3.3.6.tar.gz

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop-3.3.6"
os.environ["PATH"] += ":/content/hadoop-3.3.6/bin"

!hadoop version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-04-08 18:11:26--  https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 730107476 (696M) [application/x-gzip]
Saving to: ‘hadoop-3.3.6.tar.gz’

hadoop-3.3.6.tar.gz 100%[===================>] 696.28M   450KB/s    in 29m 36s 

2026-04-08 18:41:02 (401 KB/s) - ‘hadoop-3.3.6.tar.gz’ saved [730107476/730107476]

Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with che

In [8]:
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [9]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

!java -version

openjdk version "1.8.0_482"
OpenJDK Runtime Environment (build 1.8.0_482-8u482-ga~us1-0ubuntu1~22.04-b08)
OpenJDK 64-Bit Server VM (build 25.482-b08, mixed mode)


In [15]:
%%writefile weather.txt

2024-01-01,30,18,12
2024-01-02,32,20,10
2024-01-03,28,16,8
2024-01-04,35,22,14
2024-01-05,33,21,11
2024-01-06,29,17,9
2024-01-07,31,19,13

Writing weather.txt


In [21]:
%%writefile WeatherAverage.java

import java.io.IOException;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;

import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;

import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.Mapper;
import org.apache.hadoop.mapreduce.Reducer;

import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class WeatherAverage {

    public static class WeatherMapper
    extends Mapper<Object, Text, Text, IntWritable>{

        public void map(Object key, Text value, Context context)
        throws IOException, InterruptedException {

            String line = value.toString().trim();

            if(line.isEmpty()) return;

            String[] data = line.split(",");

            if(data.length < 4) return;

            int temp = Integer.parseInt(data[1]);
            int dew = Integer.parseInt(data[2]);
            int wind = Integer.parseInt(data[3]);

            context.write(new Text("Temperature"), new IntWritable(temp));
            context.write(new Text("DewPoint"), new IntWritable(dew));
            context.write(new Text("WindSpeed"), new IntWritable(wind));
        }
    }

    public static class WeatherReducer
    extends Reducer<Text,IntWritable,Text,Text>{

        public void reduce(Text key, Iterable<IntWritable> values,
        Context context) throws IOException, InterruptedException {

            int sum = 0;
            int count = 0;

            for (IntWritable val : values) {
                sum += val.get();
                count++;
            }

            double avg = (double) sum / count;

            context.write(key, new Text("Average = " + avg));
        }
    }

    public static void main(String[] args) throws Exception {

        Configuration conf = new Configuration();

        Job job = Job.getInstance(conf, "weather average");

        job.setJarByClass(WeatherAverage.class);

        job.setMapperClass(WeatherMapper.class);
        job.setReducerClass(WeatherReducer.class);

        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(IntWritable.class);

        FileInputFormat.addInputPath(job,
        new Path(args[0]));

        FileOutputFormat.setOutputPath(job,
        new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}

Overwriting WeatherAverage.java


In [22]:
!rm -f *.class
!rm -f *.jar
!rm -rf weather_output

!javac -source 8 -target 8 -classpath `hadoop classpath` WeatherAverage.java

In [23]:
!jar cf weather.jar WeatherAverage*.class

In [24]:
!hadoop jar weather.jar WeatherAverage weather_input weather_output

2026-04-08 19:00:07,783 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-04-08 19:00:07,881 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-04-08 19:00:07,881 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-04-08 19:00:08,049 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-04-08 19:00:08,267 INFO input.FileInputFormat: Total input files to process : 1
2026-04-08 19:00:08,310 INFO mapreduce.JobSubmitter: number of splits:1
2026-04-08 19:00:08,574 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local907180973_0001
2026-04-08 19:00:08,575 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-08 19:00:08,952 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-04-08 19:00:08,953 INFO mapreduce.Job: Running job: job_local907180973_00

In [25]:
!cat weather_output/part-r-00000

DewPoint	Average = 19.0
Temperature	Average = 31.142857142857142
WindSpeed	Average = 11.0
